# Session 15 · Association

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - transactions to a one-hot basket

In [2]:
import pandas as pd
from apriori import association_rules, frequent_itemsets, show, to_basket
transactions = [
    ["bread", "eggs", "milk"], ["bread", "butter", "eggs"], ["chai", "rusk", "milk"],
    ["chai", "rusk", "sugar"], ["bread", "eggs"], ["chai", "milk", "sugar"],
    ["bread", "butter", "milk"], ["chai", "rusk"], ["bread", "eggs", "butter"], ["coke", "chips"],
]
basket = to_basket(transactions)
print(basket.astype(int))

   bread  butter  chai  chips  coke  eggs  milk  rusk  sugar
0      1       0     0      0     0     1     1     0      0
1      1       1     0      0     0     1     0     0      0
2      0       0     1      0     0     0     1     1      0
3      0       0     1      0     0     0     0     1      1
4      1       0     0      0     0     1     0     0      0
5      0       0     1      0     0     0     1     0      1
6      1       1     0      0     0     0     1     0      0
7      0       0     1      0     0     0     0     1      0
8      1       1     0      0     0     1     0     0      0
9      0       0     0      1     1     0     0     0      0


## Example 2 - support by hand

In [3]:
support = basket.mean()
print(support.sort_values(ascending=False))
pair = (basket["bread"] & basket["eggs"]).mean()
print("support{bread, eggs} =", pair)

bread     0.5
chai      0.4
eggs      0.4
milk      0.4
butter    0.3
rusk      0.3
sugar     0.2
chips     0.1
coke      0.1
dtype: float64
support{bread, eggs} = 0.4


## Example 3 - confidence and lift by hand

In [4]:
conf = pair / support["bread"]
lift = conf / support["eggs"]
print(f"bread -> eggs: confidence {conf:.2f}, lift {lift:.2f}")
assert round(conf, 2) == 0.8 and round(lift, 2) == 2.0

bread -> eggs: confidence 0.80, lift 2.00


## Example 4 - frequent itemsets with our Apriori

In [5]:
fi = frequent_itemsets(basket, min_support=0.2)
print(fi)

                             itemset  support
0                 frozenset({bread})      0.5
1                  frozenset({chai})      0.4
2                  frozenset({eggs})      0.4
3                  frozenset({milk})      0.4
4           frozenset({bread, eggs})      0.4
5                frozenset({butter})      0.3
6                  frozenset({rusk})      0.3
7            frozenset({chai, rusk})      0.3
8         frozenset({bread, butter})      0.3
9                 frozenset({sugar})      0.2
10           frozenset({chai, milk})      0.2
11         frozenset({butter, eggs})      0.2
12          frozenset({chai, sugar})      0.2
13          frozenset({bread, milk})      0.2
14  frozenset({bread, eggs, butter})      0.2


## Example 5 - rules, filtered and sorted

In [6]:
rules = association_rules(fi, min_confidence=0.6)
show(rules[rules["lift"] > 1])

                  rusk -> chai           support 0.30  confidence 1.00  lift 2.50
                 sugar -> chai           support 0.20  confidence 1.00  lift 2.50
                  chai -> rusk           support 0.30  confidence 0.75  lift 2.50
                 bread -> eggs           support 0.40  confidence 0.80  lift 2.00
                  eggs -> bread          support 0.40  confidence 1.00  lift 2.00
                 bread -> butter         support 0.30  confidence 0.60  lift 2.00
                butter -> bread          support 0.30  confidence 1.00  lift 2.00
          butter, eggs -> bread          support 0.20  confidence 1.00  lift 2.00
                butter -> eggs           support 0.20  confidence 0.67  lift 1.67
                butter -> bread, eggs    support 0.20  confidence 0.67  lift 1.67


## Example 6 - substitutes (lift < 1)

In [7]:
rules_all = association_rules(fi, min_confidence=0.0)
show(rules_all[rules_all["lift"] < 1])